<a href="https://colab.research.google.com/github/YashThirani21/docagent/blob/main/notebooks/colab_notebook_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# docagent on Google Colab (vLLM), from GitHub

Run top to bottom on a GPU runtime (Runtime → Change runtime type → GPU). The code is cloned from
[github.com/YashThirani21/docagent](https://github.com/YashThirani21/docagent). **No setup or tokens needed.**

Where state that outlives the runtime (`store/` and the run outputs) goes is picked in cell 1 by `PERSIST`:
- `"auto"` (default): `"hf"` if this Colab account has an `HF_TOKEN` secret, otherwise `"none"`.
- `"none"`: nothing is saved between sessions. The index is built fresh, and the run outputs are downloaded to your
  computer as a zip at the end.
- `"drive"`: your own `MyDrive/ColabProjects/` (asks for Drive access once).
- `"hf"`: a private Hugging Face dataset repo, `<your HF username>/docagent-store`, in the account of whoever runs the
  notebook. Every save is a commit, so earlier stores and runs stay available. Needs a Colab secret `HF_TOKEN`
  (🔑 icon in the left bar) holding a token with write access.

## 1. Settings

In [1]:
import os, time
from pathlib import Path

REPO_URL = "https://github.com/YashThirani21/docagent.git"
REF = "main"
PROJECT = Path("/content/senior-data-scientist")

PERSIST = "auto"                # "auto", "none" (session only), "drive" (your MyDrive/ColabProjects) or "hf"
HF_DATASET = None               # for "hf" only: <token owner>/docagent-store, created private on the first save
DRIVE_DIR = Path("/content/drive/MyDrive/ColabProjects")

RUN_ID = time.strftime("%Y%m%d-%H%M%S")             # run outputs are saved under runs/<RUN_ID>/

if PERSIST in ("auto", "hf"):
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        from huggingface_hub import whoami
        # Each person saves to their own account: anyone else's token can't write to the maintainer's repo
        HF_DATASET = HF_DATASET or f"{whoami()['name']}/docagent-store"
        PERSIST = "hf"
    except Exception as e:      # no such secret, notebook access not granted, or the token is invalid
        if PERSIST == "hf":
            raise RuntimeError(
                f"No usable Colab secret HF_TOKEN ({type(e).__name__}). Add a token with Write access in "
                "Colab's 🔑 Secrets panel as HF_TOKEN with 'Notebook access' on, or set PERSIST = \"none\"."
            ) from None
        PERSIST = "none"
if PERSIST == "drive":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
elif PERSIST not in ("hf", "none"):
    raise ValueError(f"PERSIST must be 'auto', 'none', 'drive' or 'hf', not {PERSIST!r}")
print(f"persist: {PERSIST}" + (f" ({HF_DATASET})" if PERSIST == "hf" else "") + f", run: {RUN_ID}")

persist: none, run: 20261020-111803


## 2. Clone the project

In [2]:
import sys
if "chat" in sys.modules:
    # The running chat app keeps store/traces.db open: deleting it now sends its traces to a deleted file
    raise RuntimeError("The chat app is running from this project. Runtime → Restart session, then run from cell 1.")
!rm -rf {PROJECT}
!git clone -q {REPO_URL} {PROJECT}
%cd {PROJECT}
!git checkout -q {REF} && git log -1 --format="running %h  %s  (%cd)" --date=short

/content/senior-data-scientist
running 4d6bac7  Notebook changes  (2026-10-20)


## 3. Save / restore helpers

Every read and write of saved state goes through these three functions (no-ops or local downloads when
`PERSIST` is `"none"`), so switching `PERSIST` changes nothing
else in the notebook. Hugging Face calls run through the `hf` CLI with `HF_HUB_OFFLINE=0`, because the docagent
config puts the kernel in offline mode once it is loaded (`privacy.offline`).

In [3]:
import shutil, sqlite3, subprocess

STORE_ARCHIVE = Path("/content/store-backup.tar.gz")
RUN_ITEMS = ["outputs", "eval/results", "store/logs", "vllm.log"]   # + store/traces.db, snapshotted below
TRACES_DB = Path("store/traces.db")

def _hf(*args):
    env = {**os.environ, "HF_HUB_OFFLINE": "0"}
    return subprocess.run(["hf", *args, "--repo-type", "dataset"], env=env, capture_output=True, text=True)

def pull_store():
    """Restore store/ from the last saved backup; on the first run there is none."""
    STORE_ARCHIVE.unlink(missing_ok=True)
    if PERSIST == "none":
        print("Session-only run: starting with an empty store; cell 10 builds the index.")
        return
    if PERSIST == "hf":
        # Fresh folder: if the Hub can't be reached, `hf download` silently returns a file already in --local-dir
        pull_dir = Path("/content/hf_pull")
        shutil.rmtree(pull_dir, ignore_errors=True)
        r = _hf("download", HF_DATASET, STORE_ARCHIVE.name, "--local-dir", str(pull_dir))
        found = r.returncode == 0 and (pull_dir / STORE_ARCHIVE.name).is_file()
        if found:
            shutil.move(pull_dir / STORE_ARCHIVE.name, STORE_ARCHIVE)
        error = next((l for l in r.stderr.splitlines() if l.startswith("Error")), r.stderr.strip()[:200])
        missing = f"{HF_DATASET}/{STORE_ARCHIVE.name} ({error})"
    else:
        found = (DRIVE_DIR / STORE_ARCHIVE.name).is_file()
        if found:
            shutil.copy2(DRIVE_DIR / STORE_ARCHIVE.name, STORE_ARCHIVE)
        missing = DRIVE_DIR / STORE_ARCHIVE.name
    if not found:
        # First run: cell 10 builds the index from scratch, cell 12 saves the backup.
        print(f"No backup at {missing} - starting with an empty store (first run).")
        return
    subprocess.run(["tar", "-xzf", str(STORE_ARCHIVE), "-C", str(PROJECT)], check=True)
    print("Store restored successfully.")

def push_store():
    """Save store/ as one archive; on HF each save is a new commit of the same file."""
    if PERSIST == "none":
        print("Session-only run: store/ is not saved (it is rebuilt by cell 10 next time).")
        return
    if (PROJECT / TRACES_DB).is_file():
        # traces.db is in WAL mode: recent requests sit in traces.db-wal until checkpointed into the main file
        with sqlite3.connect(PROJECT / TRACES_DB) as con:
            con.execute("PRAGMA wal_checkpoint(FULL)")
    subprocess.run(["tar", "-czf", str(STORE_ARCHIVE), "-C", str(PROJECT), "store"], check=True)
    if PERSIST == "hf":
        r = _hf("upload", HF_DATASET, str(STORE_ARCHIVE), STORE_ARCHIVE.name, "--private",
                "--commit-message", f"store backup {RUN_ID}")
        if r.returncode:
            raise RuntimeError(r.stderr)
        where = f"hf.co/datasets/{HF_DATASET}"
    else:
        shutil.copy2(STORE_ARCHIVE, DRIVE_DIR / STORE_ARCHIVE.name)
        where = DRIVE_DIR / STORE_ARCHIVE.name
    print(f"Backup saved: {where}")
    print(f"Archive size: {STORE_ARCHIVE.stat().st_size / (1024**3):.2f} GB")

def push_run():
    """Save outputs, eval results, traces and logs under runs/<RUN_ID>/."""
    staging = Path("/content/run_out") / RUN_ID
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    for item in RUN_ITEMS:
        p = PROJECT / item
        if p.is_dir():
            shutil.copytree(p, staging / p.name)
        elif p.is_file():
            shutil.copy2(p, staging / p.name)
        else:
            print(f"skipped (not found): {item}")
    if (PROJECT / TRACES_DB).is_file():
        # A copy of the file alone misses what is still in traces.db-wal; the backup API includes it
        with sqlite3.connect(PROJECT / TRACES_DB) as src, sqlite3.connect(staging / TRACES_DB.name) as dst:
            src.backup(dst)
    if PERSIST == "hf":
        r = _hf("upload", HF_DATASET, str(staging), f"runs/{RUN_ID}", "--private",
                "--commit-message", f"run {RUN_ID}")
        if r.returncode:
            raise RuntimeError(r.stderr)
        print(f"Run saved: hf.co/datasets/{HF_DATASET}/tree/main/runs/{RUN_ID}")
    elif PERSIST == "none":
        from google.colab import files
        zip_path = shutil.make_archive(f"/content/docagent-run-{RUN_ID}", "zip", staging)
        files.download(zip_path)
        print(f"Run downloaded to your computer: {Path(zip_path).name}")
    else:
        shutil.copytree(staging, DRIVE_DIR / "run_out" / RUN_ID, dirs_exist_ok=True)
        print(f"Run saved: {DRIVE_DIR / 'run_out' / RUN_ID}")

## 4. Restore `store/` from the last backup (skipped on the first run)

In [4]:
pull_store()

Session-only run: starting with an empty store; cell 10 builds the index.


## 5. Install vLLM and the docagent package

In [5]:
%cd /content/senior-data-scientist

!pip install -q vllm==0.31.0 -e . -c requirements-gpu.txt

import subprocess, sys
r = subprocess.run([sys.executable, "-c", "import docagent, vllm; print('docagent and vllm', vllm.__version__, 'installed')"],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("Install failed; rerun this cell without -q to see pip's error.\n" + r.stderr[-1500:])
print(r.stdout.strip())

/content/senior-data-scientist
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

## 6. Check the GPU

In [6]:
import shutil, subprocess
# vLLM and the bge-m3 embedder (device: cuda in the colab configs) both need a GPU; stop before cells 7-10 fail
if not shutil.which("nvidia-smi") or subprocess.run(["nvidia-smi"], capture_output=True).returncode:
    raise RuntimeError("No GPU in this session. Runtime → Change runtime type → T4 GPU (or L4/A100), "
                       "then Runtime → Run all. If no GPU is offered, Colab's free GPU quota is used up for now.")
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
NVIDIA L4, 23034 MiB, 0 MiB


## 7. Pick the config and start the matching vLLM server

In [7]:
import os, yaml
%cd /content/senior-data-scientist

# Pick the agent version here; the vLLM server is started to match it.
CONFIG = "configs/colab_vllm_awq.yaml"   # or "configs/colab_vllm.yaml"
# CONFIG = "configs/colab_vllm.yaml"
MAX_LEN = 8192                           # vLLM --max-model-len (16384 if prompts get rejected for length)
GPU_UTIL = 0.90                          # share of GPU memory for vLLM; the rest is for bge-m3 on cuda

os.environ["DOCAGENT_CONFIG"] = CONFIG   # CLI, chat app and Agent() all read this
SERVED = yaml.safe_load(open(CONFIG))["llm"]["model"]   # vLLM must serve exactly the name the config asks for

print(SERVED)
if SERVED.endswith("-awq"):
    # AWQ 4-bit weights in fp16: fits a T4
    print(f"{CONFIG}: serving AWQ as {SERVED}")
    !MAX_LEN={MAX_LEN} GPU_UTIL={GPU_UTIL} MODEL=Qwen/Qwen2.5-VL-7B-Instruct-AWQ QUANT=awq DTYPE=half SERVED={SERVED} bash scripts/start_vllm.sh
else:
    # start_vllm.sh picks the weights for the GPU: bf16 on L4/A100, AWQ on a T4
    print(f"{CONFIG}: serving as {SERVED}, weights chosen by GPU")
    !MAX_LEN={MAX_LEN} GPU_UTIL={GPU_UTIL} SERVED={SERVED} bash scripts/start_vllm.sh

/content/senior-data-scientist
qwen2.5-vl-7b-awq
configs/colab_vllm_awq.yaml: serving AWQ as qwen2.5-vl-7b-awq
GPU 23034 MB, compute 89: serving Qwen/Qwen2.5-VL-7B-Instruct-AWQ as qwen2.5-vl-7b-awq (quant=awq dtype=half util=0.9)
vLLM starting (pid 6089), log: vllm.log
vLLM ready


## 8. Check the server responds

In [8]:
!curl -s http://127.0.0.1:8000/v1/models | head -c 200

{"object":"list","data":[{"id":"qwen2.5-vl-7b-awq","object":"model","created":1792495683,"owned_by":"vllm","root":"Qwen/Qwen2.5-VL-7B-Instruct-AWQ","parent":null,"max_model_len":8192,"permission":[{"i

## 9. Download the embedding model (bge-m3)

In [9]:
!python scripts/download_models.py

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0% 0/30 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/191 [00:00<?, ?B/s]           

Fetching 30 files:   3% 1/30 [00:00<00:07,  3.67it/s]
Reconstructing (incomplete total...):   3% 191/6.34k [00:00<00:07, 772B/s]
Reconstructing (incomplete total...):  29% 6.34k/22.2k [00:00<00:20, 772B/s]
Reconstructing (incomplete total...):   4% 6.34k/154k [00:00<03:11, 772B/s] 
Reconstructing (incomplete total...):  14% 22.2k/155k [00:00<02:51, 772B/s]
Reconstructing (incomplete total...):  15% 22.8k/156k [00:00<02:52, 772B/s]
Reconstructing (incomplete total...):  16% 24.5k/156k [00:00<02:50, 772B/s]
Reconstructing (incomplete total...):   7% 156k/2.26M [00:00<45:19, 772B/s]
Reconstructing (incomplete total...):   6% 156k/2.42M [00:00<48:44, 7

## 10. Ingest the reports

In [10]:
!python -m docagent.cli ingest data/reports
!python -m docagent.cli status

Loading weights: 100% 391/391 [00:00<00:00, 2748.17it/s]
pre tokenize: 100% 1/1 [00:00<00:00, 446.63it/s]
Inference Embeddings: 100% 1/1 [00:00<00:00, 44.95it/s]
/content/senior-data-scientist/src/docagent/store.py:46: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self.client.create_payload_index(self.collection, field, qm.PayloadSchemaType.KEYWORD)
pre tokenize: 100% 2/2 [00:00<00:00, 254.07it/s]
Inference Embeddings: 100% 2/2 [00:00<00:00,  3.25it/s]
pre tokenize: 100% 4/4 [00:00<00:00, 228.65it/s]
Inference Embeddings: 100% 4/4 [00:00<00:00,  4.17it/s]
ntsb_16-006.pdf: 26 chunks (12 figures)
ntsb_18-061.pdf: 64 chunks (33 figures)
ntsb_24-013.pdf: 6 chunks (5 figures)
collection reports_bge-m3_v1_bf16desc: 96 chunks
  ntsb_16-006: 26
  ntsb_18-061: 64
  ntsb_24-013: 6


## 11. Launch the chat app (public Gradio share link)

In [11]:
import sys
sys.path.insert(0, "/content/senior-data-scientist/src")
sys.path.insert(0, "/content/senior-data-scientist/app")

import chat
from IPython.display import HTML, display

demo = chat.demo

demo.launch(
    server_name="127.0.0.1",
    server_port=7860,
    share=True,
    prevent_thread_lock=True,
    inline=False,
)

url = demo.share_url
display(HTML(f'<a href="{url}" target="_blank">{url}</a>'))

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cb46f1a422f17e8061.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 12. Save `store/` (Hugging Face or Drive, per `PERSIST`; skipped when `"none"`)

In [12]:
push_store()

Session-only run: store/ is not saved (it is rebuilt by cell 10 next time).


## 13. Save run outputs, eval results, traces and logs

In [13]:
push_run()

skipped (not found): outputs


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Run downloaded to your computer: docagent-run-20261020-111803.zip


## 14. Ask a few questions in the notebook

In [17]:
import sys, time
sys.path.insert(0, "/content/senior-data-scientist/src")
from IPython.display import Markdown, display
from docagent.agent import Agent
from docagent.render import render_markdown

questions = [
    "What is the fatigue life of a 2024-T3 spar in cycles?",
    "Why did the left wing main spar fail?",
    "What caused the throttle arm to fracture?",
]
agent = chat.agent if "chat" in globals() else Agent()
batch_id = "questions-" + time.strftime("%Y%m%d-%H%M%S")
for q in questions:
    resp = agent.ask(q, batch_id=batch_id)
    display(Markdown(f"### {q}\n\n" + render_markdown(resp)))
print("traces:", batch_id)

request dd8b87cfacc6  succeeded  3.4s  created 2026-10-20T11:34:08.194Z  batch=questions-20261020-113339
  text:     What caused the throttle arm to fracture?
  versions: {"router": "rules+vlm-v1", "drawing": "drawing-geometry-v1", "report_qa": "report-qa-v3", "llm": "qwen2.5-vl-7b-awq"}  config=eb400b779831
  llm:      1 model calls, tokens 2679 in / 116 out
  outcome:  {"routes": ["report_qa"], "errors": [], "drawings": [], "answer": "The throttle arm fractured due to bending overstress.", "grounded": true, "citations": ["ntsb_24-013 p3", "ntsb_24-013 p4 fig2", "ntsb_24-013 p6 fig4", "ntsb_24-013 p6 fig5", "ntsb_24-013 p5 fig3", "ntsb_24-013 p4 fig1", "ntsb_16-...
spans:
  router  0.00s  {"n_attachments": 0, "route": "report_qa", "tier": "rule"}
  report_qa  3.39s  {"has_text": true, "n_images": 0}
    qa.retrieve  0.08s  {"query": "What caused the throttle arm to fracture?", "retriever": "hybrid-rrf-v3-textfirst-reg-focus-next...
    qa.generate  3.24s  {"n_sources": 11, "n_images":

## 15. Spans of one request (the latest by default)

In [19]:
!python -m docagent.cli trace last

request dd8b87cfacc6  succeeded  3.4s  created 2026-10-20T11:34:08.194Z  batch=questions-20261020-113339
  text:     What caused the throttle arm to fracture?
  versions: {"router": "rules+vlm-v1", "drawing": "drawing-geometry-v1", "report_qa": "report-qa-v3", "llm": "qwen2.5-vl-7b-awq"}  config=eb400b779831
  llm:      1 model calls, tokens 2679 in / 116 out
  outcome:  {"routes": ["report_qa"], "errors": [], "drawings": [], "answer": "The throttle arm fractured due to bending overstress.", "grounded": true, "citations": ["ntsb_24-013 p3", "ntsb_24-013 p4 fig2", "ntsb_24-013 p6 fig4", "ntsb_24-013 p6 fig5", "ntsb_24-013 p5 fig3", "ntsb_24-013 p4 fig1", "ntsb_16-...
spans:
  router  0.00s  {"n_attachments": 0, "route": "report_qa", "tier": "rule"}
  report_qa  3.39s  {"has_text": true, "n_images": 0}
    qa.retrieve  0.08s  {"query": "What caused the throttle arm to fracture?", "retriever": "hybrid-rrf-v3-textfirst-reg-focus-next...
    qa.generate  3.24s  {"n_sources": 11, "n_images":

## 16. Trace of the latest request

---



In [16]:
import sqlite3, pandas as pd

con = sqlite3.connect(PROJECT / "store/traces.db")
REQUEST_ID = pd.read_sql("select request_id from requests order by created_at desc limit 1", con).iloc[0, 0]
# REQUEST_ID = "53b8c2c82def"   # or any id from a chat reply / `docagent requests`
print("request", REQUEST_ID)
pd.read_sql("select * from spans where request_id = ? order by started_at", con, params=(REQUEST_ID,))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 42.85it/s]


### What is the fatigue life of a 2024-T3 spar in cycles?

The fatigue life of a 2024-T3 spar cannot be determined from the provided information. The reports focus on the examination of the spar after the accident, detailing the presence of fatigue features and the origin of these fatigue cracks. However, they do not provide data on the fatigue life of the material in terms of cycles. The reports mention the hardness and conductivity of the spar, but these properties are not directly related to fatigue life. Additionally, the reports do not provide any information on the cyclic loading history of the spar during its service life.

**Sources:** ntsb_18-061 p.1; ntsb_18-061 p.7; ntsb_18-061 p.4

<sub>request 329e6a066dc6 · routes: report_qa (rule)</sub>

### Why did the left wing main spar fail?

The left wing main spar failed due to fatigue cracking originating from the corner radius between the hole LC-1 bore and the main spar lower surface. This fatigue cracking extended up to 0.119 inch from the hole surface, consistent with fatigue features observed under SEM examination.

**Sources:** ntsb_18-061 p.7; ntsb_18-061 p.26 (Fig. 20)

<sub>request edd7fb08e59c · routes: report_qa (rule)</sub>

### What caused the throttle arm to fracture?

The throttle arm fractured due to bending overstress.

**Sources:** ntsb_24-013 p.3; ntsb_24-013 p.4 (Fig. 2); ntsb_24-013 p.6 (Fig. 4); ntsb_24-013 p.6 (Fig. 5); ntsb_24-013 p.5 (Fig. 3); ntsb_24-013 p.4 (Fig. 1); ntsb_16-006 p.6 (Fig. 1)

<sub>request dd8b87cfacc6 · routes: report_qa (rule)</sub>

traces: questions-20261020-113339


## Reading saved runs back

On Hugging Face, fetch any earlier run, or the store as it was at any commit (see the repo's *Files → History*):
```python
!HF_HUB_OFFLINE=0 hf download {HF_DATASET} --repo-type dataset --include "runs/<RUN_ID>/*" --local-dir /content/saved
!HF_HUB_OFFLINE=0 hf download {HF_DATASET} store-backup.tar.gz --repo-type dataset --revision <commit> --local-dir /content
```
With `"none"` each run is the downloaded `docagent-run-<RUN_ID>.zip`. On Drive they are in `MyDrive/ColabProjects/run_out/<RUN_ID>/`.